In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=42)

In [3]:
df = pd.read_excel(
    "../../../data/bpic20_Dom.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Amount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:Amount,concept:name,org:resource,org:role,time_delta
0,declaration 100000,2018-01-30 09:20:07,600.844116,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 100000,2018-02-07 09:58:46,600.844116,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,693519.0
2,declaration 100000,2018-02-08 10:59:05,600.844116,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,90019.0
3,declaration 100000,2018-02-09 12:42:49,600.844116,Request Payment,SYSTEM,UNDEFINED,92624.0
4,declaration 100000,2018-02-12 17:31:20,600.844116,Payment Handled,SYSTEM,UNDEFINED,276511.0
5,declaration 100005,2018-01-30 09:38:54,35.133686,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
6,declaration 100005,2018-01-30 09:38:57,35.133686,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0
7,declaration 100005,2018-01-30 10:04:10,35.133686,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,1513.0
8,declaration 100005,2018-01-31 12:45:18,35.133686,Request Payment,SYSTEM,UNDEFINED,96068.0
9,declaration 100005,2018-02-01 17:31:17,35.133686,Payment Handled,SYSTEM,UNDEFINED,103559.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Amount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [2.00, 284911.00]                        42289.5000 quantile_derived    
case:Amount                    continuous     case     yes    [6.91, 219.03]                           25.3885    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
org:role                       categorical    event    ye

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load()

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../pretrained_models/"
)

In [15]:
engine.parallel_sets

[]

In [16]:
engine.branching_sets

[{'Declaration APPROVED by ADMINISTRATION',
  'Declaration APPROVED by BUDGET OWNER',
  'Declaration APPROVED by PRE_APPROVER',
  'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE'},
 {'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR'},
 {'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by MISSING'},
 {'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR'}]

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Dom-cf_generated_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/220 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 142992,4,1,0,0.018574,0.037148,0.0,0.428571,0.272727,...,1.270420,0.272727,0.018574,0.0,0.037148,0.428571,0.550547,0.550547,0.999994,0.999994
1,0,declaration 115669,4,1,0,0.006727,0.013454,0.0,0.428571,0.272727,...,1.382459,0.272727,0.006727,0.0,0.013454,0.428571,0.674433,0.674433,0.999999,0.999999
2,0,declaration 138710,4,1,0,0.052286,0.104573,0.0,0.428571,0.272727,...,1.376665,0.272727,0.052286,0.0,0.104573,0.428571,0.623080,0.623080,0.999998,0.999998
3,0,declaration 141310,4,1,0,0.006727,0.013454,0.0,0.428571,0.272727,...,1.382400,0.272727,0.006727,0.0,0.013454,0.428571,0.674374,0.674374,0.999999,0.999999
4,0,declaration 113587,5,1,0,0.007311,0.014621,0.0,0.285714,0.230769,...,1.074342,0.230769,0.007311,0.0,0.014621,0.285714,0.550547,0.550547,0.999994,0.999994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,20,declaration 124561,11,1,0,0.178907,0.357814,0.0,0.360000,0.400000,...,0.938907,0.400000,0.178907,0.0,0.357814,0.360000,0.000000,0.124572,0.000000,0.000000
148,20,declaration 134394,11,1,0,0.095081,0.190161,0.0,0.360000,0.840000,...,2.250610,0.840000,0.095081,0.0,0.190161,0.360000,0.955530,0.355962,1.000000,0.000000
149,20,declaration 129484,11,1,0,0.058985,0.117970,0.0,0.320000,0.400000,...,0.778985,0.400000,0.058985,0.0,0.117970,0.320000,0.000000,0.000000,0.000000,0.000000
150,20,declaration 126499,11,1,0,0.059475,0.118949,0.0,0.360000,0.840000,...,2.214792,0.840000,0.059475,0.0,0.118949,0.360000,0.955318,0.006179,1.000000,0.000000


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/220 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 142992,4,1,0,0.018574,0.037148,0.0000,0.428571,0.272727,...,1.270420,0.272727,0.018574,0.0000,0.037148,0.428571,0.550547,0.550547,0.999994,0.999994
1,0,declaration 115669,4,1,0,0.073405,0.146810,0.0000,0.428571,0.272727,...,1.325251,0.272727,0.073405,0.0000,0.146810,0.428571,0.550547,0.550547,0.999994,0.999994
2,0,declaration 138710,4,1,0,0.047589,0.095177,0.0000,0.428571,0.272727,...,1.372148,0.272727,0.047589,0.0000,0.095177,0.428571,0.623261,0.623261,0.999998,0.999998
3,0,declaration 141310,4,1,0,0.067748,0.135495,0.0000,0.428571,0.272727,...,1.319594,0.272727,0.067748,0.0000,0.135495,0.428571,0.550547,0.550547,0.999994,0.999994
4,0,declaration 113587,5,1,0,0.007311,0.014621,0.0000,0.285714,0.230769,...,1.074342,0.230769,0.007311,0.0000,0.014621,0.285714,0.550547,0.550547,0.999994,0.999994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,20,declaration 124561,11,1,0,0.394730,0.351961,0.4375,0.640000,0.400000,...,1.665379,0.400000,0.394730,0.4375,0.351961,0.640000,0.230649,0.000000,0.000000,0.000000
148,20,declaration 134394,11,1,0,0.271247,0.229995,0.3125,0.560000,0.840000,...,2.626800,0.840000,0.271247,0.3125,0.229995,0.560000,0.955553,0.000000,1.000000,0.000000
149,20,declaration 129484,11,1,0,0.060746,0.121492,0.0000,0.320000,0.400000,...,0.780746,0.400000,0.060746,0.0000,0.121492,0.320000,0.000000,0.000000,0.000000,0.000000
150,20,declaration 126499,11,1,0,0.087740,0.175479,0.0000,0.360000,0.840000,...,2.243292,0.840000,0.087740,0.0000,0.175479,0.360000,0.955553,0.000000,1.000000,0.000000


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()